# NanoVision: CNT Feature Extraction

This notebook demonstrates how to use NanoVision programmatically to extract
morphological features from CNT prediction annotations.

Unlike the runner-based inference/evaluation workflow, this example directly
imports and calls a reusable NanoVision Python API.

Workflow:

1. Configure the prediction COCO JSON.
2. Configure an optional output directory.
3. Validate the input.
4. Extract object-level and image-level CNT features.
5. Inspect the resulting tables.
6. Optionally save the extracted features as CSV files.

In [ ]:
from pathlib import Path

import cnt_project

from cnt_project.features.pipelines.prediction_feature_extraction import (
    extract_features_from_prediction_json,
)

print("NanoVision package:")
print(cnt_project.__file__)

CNTLib package:
C:\Users\abd93000\PycharmProjects\cnt_project_review\src\cnt_project\__init__.py


## Configuration

Configure the prediction JSON and feature-output directory below.

`PREDICTION_JSON` can point to any compatible CNT prediction COCO JSON.
It does not need to have been generated by NanoVision in the current project.

`OUTPUT_DIR` specifies where the feature-extraction artifacts will be written.

In [2]:
# ------------------------------------------------------------------
# User configuration
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

GLOBAL_OUTPUTS_ROOT = (
    PROJECT_ROOT
    / "global_outputs"
)

RUN_NAME = "cntlib_notebook_inference_example"

PREDICTION_JSON = (
    GLOBAL_OUTPUTS_ROOT
    / "runs"
    / RUN_NAME
    / "inference"
    / "predicted_annotations_poly.json"
)

OUTPUT_DIR = (
    GLOBAL_OUTPUTS_ROOT
    / "runs"
    / RUN_NAME
    / "eval"
    / "features"
)

DEFAULT_IMAGE_HEIGHT = 256
DEFAULT_IMAGE_WIDTH = 256

SAVE_CSV = True

### Validate configuration

In [3]:
PREDICTION_JSON = PREDICTION_JSON.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()

print("Prediction JSON:")
print(PREDICTION_JSON)

print("\nOutput directory:")
print(OUTPUT_DIR)

if not PREDICTION_JSON.exists():
    raise FileNotFoundError(
        f"Prediction JSON does not exist: {PREDICTION_JSON}"
    )

if DEFAULT_IMAGE_HEIGHT <= 0 or DEFAULT_IMAGE_WIDTH <= 0:
    raise ValueError(
        "DEFAULT_IMAGE_HEIGHT and DEFAULT_IMAGE_WIDTH must be positive."
    )

if SAVE_CSV:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

print("\nConfiguration is valid.")

Prediction JSON:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\inference\predicted_annotations_poly.json

Output directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\eval\features

Configuration is valid.


## Run Feature Extraction

The feature-extraction pipeline reads the prediction annotations and writes
the resulting feature tables and, when requested, histogram figures to the
configured output directory.
there is a runner you can call but for a consumer of the library you do not always need a runner. NanoVision functionality can be embedded directly in Python. we have pipelines.

In [4]:
object_df, image_df = extract_features_from_prediction_json(
    PREDICTION_JSON,
    default_image_height=DEFAULT_IMAGE_HEIGHT,
    default_image_width=DEFAULT_IMAGE_WIDTH,
)

## Inspect Generated Outputs

In [5]:
print("Object-level features:")
print(f"Rows: {len(object_df)}")
print(f"Columns: {len(object_df.columns)}")

display(object_df.head())

print("Image-level features:")
print(f"Rows: {len(image_df)}")
print(f"Columns: {len(image_df.columns)}")

display(image_df.head())

Object-level features:
Rows: 3000
Columns: 17


,image_id,image_file,annotation_id,object_index,area_pixels2,area_um2,perimeter,perimeter_um,length,length_um,tree_length,tree_length_um,width,width_um,aspect_ratio,orientation_angle,n_connected_components
0,1293-22_3_0_20230306224436,1293-22_3_0_20230306224436.tif,1,0,13.0,0.004959,16.969785,0.331441,7.242641,0.141458,5.828427,0.113836,2.118347,0.041374,3.419006,27.107062,1.0
1,1293-22_3_0_20230306224436,1293-22_3_0_20230306224436.tif,2,1,43.0,0.016403,49.893920,0.974491,24.313708,0.474877,23.899495,0.466787,2.037656,0.039798,11.932196,18.489340,1.0
2,1293-22_3_0_20230306224436,1293-22_3_0_20230306224436.tif,3,2,78.0,0.029755,84.523604,1.650852,42.041631,0.821126,41.627417,0.813035,2.207107,0.043108,19.048299,25.320389,1.0
3,1293-22_3_0_20230306224436,1293-22_3_0_20230306224436.tif,4,3,71.0,0.027084,76.899660,1.501946,39.556349,0.772585,37.727922,0.736873,2.069036,0.040411,19.118255,12.687038,1.0
4,1293-22_3_0_20230306224436,1293-22_3_0_20230306224436.tif,5,4,62.0,0.023651,68.069711,1.329487,33.899495,0.662100,33.485281,0.654009,2.051777,0.040074,16.522020,5.389139,1.0


Image-level features:
Rows: 30
Columns: 27


,image_id,image_file,n_objects,cnt_density_per_um2,line_density_mean,line_density_std,line_density_max,nematic_order_parameter,nematic_director_angle_deg,von_mises_mean_orientation_deg,...,mean_perimeter_um,mean_length,mean_length_um,mean_tree_length,mean_tree_length_um,mean_width,mean_width_um,mean_aspect_ratio,mean_orientation_angle,mean_n_connected_components
0,1293-22_3_0_20230306224436,1293-22_3_0_20230306224436.tif,154,6.16,6.566406,2.354244,16.0,0.563404,5.116601,5.116601,...,1.028610,25.482889,0.497713,25.199245,0.492173,2.104165,0.041097,11.856602,2.781370,1.045455
1,400-1093-w11-c1p1-befo3-10sp-15flow_right,400-1093-w11-c1p1-befo3-10sp-15flow_right.tif,186,7.44,9.484375,2.623465,16.0,0.538832,13.371008,13.371008,...,1.196331,28.753248,0.561587,27.948893,0.545877,3.289732,0.064253,8.294196,10.134430,1.005376
2,400-1093-w17-c1p3-20sp-03flow_left,400-1093-w17-c1p3-20sp-03flow_left.tif,471,18.84,25.675781,5.173584,41.0,0.205023,43.248534,43.248534,...,0.889778,20.446270,0.399341,20.873938,0.407694,2.738994,0.053496,7.044531,8.718618,1.004246
3,400-1293-17-c10k1r5_pd_sp0_9_fl0_1_left,400-1293-17-c10k1r5_pd_sp0_9_fl0_1_left.tif,11,0.44,0.601562,0.732503,3.0,0.105328,57.954549,57.954549,...,0.750247,17.356717,0.338998,15.933366,0.311199,3.113175,0.060804,5.496744,-16.621785,1.000000
4,400-1293-17-c11k1r5_pd_sp1_2_fl0_1_left,400-1293-17-c11k1r5_pd_sp1_2_fl0_1_left.tif,13,0.52,0.542969,0.784437,3.0,0.216868,9.306149,9.306149,...,0.721585,16.387429,0.320067,14.190787,0.277164,3.370221,0.065825,4.759378,-5.559625,1.000000


In [6]:
if SAVE_CSV:
    object_csv = OUTPUT_DIR / "predicted_object_features.csv"
    image_csv = OUTPUT_DIR / "predicted_image_features.csv"

    object_df.to_csv(
        object_csv,
        index=False,
    )

    image_df.to_csv(
        image_csv,
        index=False,
    )

    print("Saved:")
    print(object_csv)
    print(image_csv)

Saved:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\eval\features\predicted_object_features.csv
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\eval\features\predicted_image_features.csv


In [7]:
if OUTPUT_DIR.exists():
    generated_files = [
        path
        for path in sorted(OUTPUT_DIR.rglob("*"))
        if path.is_file()
    ]

    print(f"Generated {len(generated_files)} file(s):")

    for path in generated_files:
        print(
            " -",
            path.relative_to(OUTPUT_DIR),
        )

Generated 2 file(s):
 - predicted_image_features.csv
 - predicted_object_features.csv
